In [ ]:
'''
Author: Ivan Gvozdanovic
Date: 10/19/2023

Finding optimal tour in Traveling Salesman Problem, using tabular Q-learning algorithm.

''';

In [ ]:
'''
The code works for small instances of TSP. Now check whether the policy is valid for different starting points. 
''';

In [ ]:
import scipy
import scipy.io
from datetime import date, time, datetime as Date, time, datetime
from scipy import optimize
import networkx as nx
import ast
import random
import os
import re
import numpy as np
import time as Time
from operator import itemgetter
import math as m
import copy as cpy
import matplotlib.pyplot as plt
import os
import itertools
import pickle

from TabularTspEdgeSwapENV import PolytopeENV as Env


from Q_learning import Q_learning, EdgeSwap_Q_learning


from optimal_policy_extraction import edge_swap_policy_evaluation


from draw_fiber_graph import draw_graph_animation

from create_initial_solutions import create_single_TSP_initial_solution,\
                                     create_power_grid_line_initial_solution

from reward_functions import reward_cost, calculate_reward1, calculate_reward2, calculate_reward3


from helper_functions import create_tsp_polytope_graph, extract_distance_matrix, create_state_graph, create_state_edges


$\Large \textbf{Initial Solution}$

In [ ]:



objective_table = [] # cost vector
initial_states = {} # dictionary holding the initial states.
reward_lists = [] # list holding the rewrds for each edge in the subproblem. We read it off to compute the travel cost.

patches = 1
nodes_per_patch = 6

#Pick the file to the problem:
file = 'TSP_MultiDiscrete_DQN'


available_actions, initial_states, distance_matrix, objective_table, reward_list = create_tsp_polytope_graph(nodes_per_patch, 
                                                                                                             patches, 
                                                                                                             initial_states, 
                                                                                                             reward_lists,
                                                                                                             file,
                                                                                                             reward_cost,
                                                                                                             1,100)


# available_actions, initial_states, distance_matrix, objective_table, reward_list, start_nodes_dict, end_nodes_dict, connecting_edges \
#                     = create_power_grid_line_initial_solution(2, nodes_per_patch, 1, 100)

print(len(initial_states[0]),initial_states[0])
print(available_actions)

[(0, 1), (0, 2), (1, 5), (2, 4), (5, 3)]

In [ ]:
from exact_solution import TSP_integer_program

tour_edges = TSP_integer_program(distance_matrix) # solve TSP-MZT integer program.
print("Tour edges: \n", tour_edges)

tour_edges = [(min(e),max(e)) for e in tour_edges]
or_reward = 0
for e in tour_edges:
    or_reward += objective_table[e]
print("Reward: ", or_reward)

$\Large \textbf{Main training loop}$

In [ ]:
save_data = True  # save Q table data and cost vector data.
save_plots = True  # save the plots


# Model Parameters
epsilon = 1  # exploration parameter.
reward_parameter = 1
lr = 0.07  # learning rate.
discount_factor = 0.5  # discount parameter for the reward.
episode_numbers = [10000]  # number of episodes we run the algorithm on.
path_numbers = [20] # number of paths we run for each episode.
max_path_lengths = [5] # maximum number of steps allowed per a path.
table_size = len(available_actions[0]) # the size of each state and action vector.

n_step_lookup = 1 #max_path_lengths[0]#10

# Set the correct Q-learning configuration.
episode_num = episode_numbers[0]
path_num = path_numbers[0]
show_path_num = path_num*50
max_path_length = max_path_lengths[0]


combinations = list(itertools.combinations(range(nodes_per_patch), 2))
action_space_values = [list(pair) for pair in combinations]
action_space_size = nodes_per_patch-1

best_states_size = 10
best_states = {0: (initial_states[0], reward_cost(reward_list, initial_states[0]))}
print(best_states)

print(action_space_values)

In [ ]:
 
# Convert dictionary values to a list of arrays
visited_states = [np.array(initial_states[0])]
visited_states = np.stack(visited_states)

#Initialize the environment.
env = Env(initial_states[0], # initial_state
         reward_list, # edge_weights
         episode_num, # total_episodes
         max_path_length,
         50, # show_path_num
         visited_states,  # visited_states
         available_actions, # basis_moves
         nodes_per_patch, # node_num
         0, # P
         best_states,
         best_states_size,
         objective_table,
         False,
         discount_factor,
         reward_function = reward_cost
         )

In [ ]:
start_time = Time.time()

Q, agent_paths, ave_episode_reward = EdgeSwap_Q_learning(epsilon, 
                                                   episode_num, 
                                                   path_num, 
                                                   table_size, 
                                                   max_path_length, 
                                                   discount_factor, 
                                                   env, 
                                                   lr, 
                                                   save_plots, 
                                                   nodes_per_patch,
                                                   action_space_values,
                                                   action_space_size,
                                                   n_step_lookup)



#Save the Q table.
if save_data:
    time = Time.localtime()
    current_time = Time.strftime("%H-%M-%S", time)
    date = datetime.now()
    d = date.isoformat()[0:10]
    data_save = [Q]
    data_save = np.array(data_save, dtype=object)
    print(d[0:10])
    np.save('Models/'
            +'Q_EP_'
            +str(episode_num)+'_P_'+str(path_num)
            +'_PL_'+ str(max_path_length)+'_Date_'+d+'_.npy',data_save)
    

    
end_time = Time.time()

print(f'It took {(end_time-start_time)/60} minutes to run {episode_num} episodes.')



$\Large \textbf{Examine the optimal policy}$

In [ ]:
data_load = np.load('Models/'
                    +'Q_EP_'
                    +str(episode_num)
                    +'_P_'+str(path_num)
                    +'_PL_'
                    + str(max_path_length)
                    +'_Date_2025-05-18'
                    +'_.npy',allow_pickle=True)
Q = data_load[0] 

path_rewards = {}

with open('Models' + os.sep + 'visited_states.pkl', 'rb') as f:
    starting_states = pickle.load(f)
print(np.array([initial_states[0]]))
trajectory_num = 3

random_initial_states = starting_states[np.random.choice(starting_states.shape[0], trajectory_num, replace=False)]
random_initial_states = np.concatenate((np.array([initial_states[0]]), random_initial_states), axis=0)


for t in range(trajectory_num):
    print("\n")
    print("####################################################################################################")
    print("####################################################################################################")
    print("####################################################################################################")
    print("####################################################################################################")
    print("####################################################################################################")
    print("\n")
    #Initialize the environment.
    env = Env(random_initial_states[t,:], # initial_states[0]initial_state
             reward_list, # edge_weights
             episode_num, # total_episodes
             max_path_lengths[0],
             50, # show_path_num
             starting_states,  # visited_states
             available_actions, # basis_moves
             nodes_per_patch, # node_num
             0, # P
             best_states,
             best_states_size,
             objective_table,
             True,
             discount_factor,
             reward_function = reward_cost
             )

    path_reward \
    = edge_swap_policy_evaluation(Q,env,reward_list,max_path_lengths[0],action_space_values,action_space_size, nodes_per_patch)
    path_rewards[t] = path_reward



In [ ]:
    
for t in range(trajectory_num):
    improving_state_rewards = [tup[0] for tup in path_rewards[t]]
    plt.plot(improving_state_rewards, '-o', color='black')
    plt.title("Approximate imporving paths")
    plt.xlabel("State at time t")
    plt.ylabel("State Cost")
    print(improving_state_rewards)
plt.savefig('Figures/ImprovingPath4.png')
plt.show()